# Reading in ERA-5 Data and Comparing with GridRad

## Overview

In the last notebook, we read in and analyzed radar reflectivity and azimuthal shear data from GridRad-Severe over the Turin, NY supercell on August 7th, 2023. In this follow-up notebook, we will look at a couple more variables centered around this severe thunderstorm event, this time using the ERA-5 hourly single-level data, which is part of the fifth-generation ECMWF reanalysis provided by Destination Earth's Earth Data Hub. We will then compare these graphs to the GridRad data plotted in the previous notebook and see if they match up with actual observations. 

1. Prerequisites
2. Imports
3. Content
4. Summary
5. Conclusions
6. Resources and References

## Prerequisites 

| Concepts | Importance | Notes |
| --- | --- | --- |
| [Intro to Numpy](https://foundations.projectpythia.org/core/numpy/) | Necessary | |
| [Intro to Cartopy](https://foundations.projectpythia.org/core/cartopy/cartopy) | Necessary | |
| [Understanding of Datetime](https://foundations.projectpythia.org/core/datetime/) | Helpful | Using times and calendars in Python |
| [Using MetPy](https://unidata.github.io/MetPy/latest/tutorials/unit_tutorial.html) | Helpful | Working with Units and Making Conversions |
| [Understanding of Xarray](https://foundations.projectpythia.org/core/xarray/) | Helpful | Analysis of gridded datasets |
| [Understanding of Matplotlib](https://foundations.projectpythia.org/core/matplotlib/) | Helpful | Creating plots in Python |

- **Time to learn**: About 1 hour (10 minutes per subsection)

## Imports

As always, we will start off my importing any relevant datasets we will need in this notebook. We use xarray, numpy, cartopy and matplotlib like in the last notebook, but in addition, we will need the datetime library to read in specific dates and make a time string, and metpy to make unit conversions and calculations. 

In [ ]:
import xarray as xr
import numpy as np
from datetime import datetime as dt
from metpy.units import units
import metpy.calc as mpcalc
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt

## Content #1: Importing ERA5 Pressure Level Data 

We will first use xarray to access the ERA5 hourly data on pressure levels, accessible via an Analysis Ready, Cloud Optimized (ARCO) format on Destination Earth's Earth Data Hub. We will point the object to the Zarr engine: 

In [ ]:
%%time 

ds = xr.open_dataset(
    "https://data.earthdatahub.destine.eu/era5/reanalysis-era5-pressure-levels-v0.zarr",
    storage_options={"client_kwargs":{"trust_env":True}},
    chunks={},
    engine="zarr",
)

print(f'size: {ds.nbytes / (1024 ** 4)} TB')

Next, let's specify the region and time of interest. For this, we will use the same date/time we used in Notebook 1 to analyze the same storm and environment. 

In [ ]:
lonW = -77.7
lonE = -73.7
latS = 42.6
latN = 44.6
cLat, cLon = (latS + latN)/2, (lonW + lonE)/2 

# In ERA5, longitudes run between 0 and 360, not -180 and 180:
if (lonW < 0 ):
    lonW = lonW + 360
if (lonE < 0 ):
    lonE = lonE + 360
    
expand = 1
latRange = np.arange(latS - expand,latN + expand,.25) # Expanding data range a bit beyond the plot range
lonRange = np.arange((lonW - expand),(lonE + expand),.25) # Need to match longitude values to those of the coordinate variable

# Specifying date/time: We use 0000Z on 8-8-2023, which is the most recent hourly time marker around the Turin supercell. 
Year = 2023
Month = 8
Day = 8
Hour = 0
Minute = 0
dateTime = dt(Year,Month,Day,Hour)
timeStr = dateTime.strftime("%Y-%m-%d %H%M UTC")
timeStr

## Further examine the dataset: 

In [ ]:
ds

The first variable I thought would be helpful to analyze would be the vertical velocity. I retrieved the variable from the DataArray, and then further examined it. 

In [ ]:
%%time 
use_DE = True

if (use_DE):
    vert = ds['w'].sel(valid_time=dateTime,latitude=latRange,longitude=lonRange, method='nearest')
else:
    vert = ds['lagrangian_tendency_of_air_pressure'].sel(time=dateTime,latitude=latRange,longitude=lonRange)

vert

## Performing unit conversions

It can be seen that the data variable is in units of Pa/s. It would be better to convert this into hectopascals (hPa) since that is more commonly measured on meteorological maps. This is where MetPy comes in handy, as we can use a unit conversion method on the DataArray to convert it into hPa/s:

In [ ]:
vert_hpa = vert.metpy.convert_units('hPa/s')

Let's look at the updated data values:

In [ ]:
print(vert_hpa.values)

Next, we need to set a specified pressure level for this data variable. We will select 700 hPa since that is a common atmospheric level in the mid-troposphere where many weather processes occur. Also, since we are making a 2D map from 3D data, we need to convert the data variable into 2-D dimensions. We do this by using xarray's .sel function. 

In [ ]:
target_pressure = 700 * units.hPa 
vert_selected_level = vert_hpa.sel(isobaricInhPa=target_pressure, method='nearest')

print("Shape after selecting 700 hPa level:", vert_selected_level.shape) # Making sure data is in the correct dimensions. 

## Plotting the map 

We then use matplotlib functions to plot the data on a map.

In [ ]:
datacrs = ccrs.PlateCarree() # Since data is lat-lon, use the Plate Carree native projection. 

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(1, 1, 1, projection=datacrs) 

ax.set_extent([lonW, lonE, latS, latN], crs=datacrs)
ax.add_feature(cfeature.STATES.with_scale('50m'), edgecolor='black', alpha=0.6)

gl = ax.gridlines(draw_labels=True, linewidth=1, color='gray', alpha=0.5, linestyle='--')
gl.top_labels = False
gl.right_labels = False

cf = ax.contourf(vert_selected_level.longitude, vert_selected_level.latitude, vert_selected_level,
                 levels=15, cmap='coolwarm', transform=datacrs)

cbar = fig.colorbar(cf, ax=ax, orientation='vertical', 
                    label=f'Vertical Velocity ({vert_selected_level.metpy.units})') 

level_val = vert_selected_level.isobaricInhPa.item() 
ax.set_title(f'Vertical Pressure Velocity at {level_val:.0f} hPa for {timeStr}')

plt.show()

In this graph, the negative (blue) values of vertical velocity represent upward motion and rising air, and the positive (red) values indicate downward motion and sinking air. It can be seen that the areas of negative values match up with where severe thunderstorms were currently located at this time, such as around the Turin, NY area as well as farther southwest where there were other thunderstorms moving through, as there was rapid upward motion over these areas. On the other hand, the red shadings are located over areas where there was not much precpitation, or subsidence, so this graph makes sense and matches up with observations. 

## Content 2: The V-component Wind Data 

Another variable you can look at in this dataset is the V component of the wind. I wanted to look at a slightly bigger lat/lon range for this, so I re-defined the maximum and minimum lat/lon values.

In [ ]:
lonmin = -85.5
lonmax = -70.5
latmin = 39.9
latmax = 46.9
cLat, cLon = (latmin + latmax)/2, (lonmin + lonmax)/2 

# Recall that in ERA5, longitudes run between 0 and 360, not -180 and 180
if (lonmin < 0 ):
    lonmin = lonmin + 360
if (lonmax < 0 ):
    lonmax = lonmax + 360
    
expand = 1
latRange = np.arange(latmin - expand,latmax + expand,.25) # expand the data range a bit beyond the plot range
lonRange = np.arange((lonmin - expand),(lonmax + expand),.25) # Need to match longitude values to those of the coordinate variable

Year = 2023
Month = 8
Day = 8
Hour = 0
Minute = 10
dateTime = dt(Year,Month,Day,Hour)
timeStr = dateTime.strftime("%Y-%m-%d %H%M UTC")
timeStr

Then, let's read in and inspect the data variable like we did before:

In [ ]:
%%time 

if (use_DE):
    v = ds['v'].sel(valid_time=dateTime,latitude=latRange,longitude=lonRange, method='nearest')
else:
    v = ds['northward_wind'].sel(time=dateTime,latitude=latRange,longitude=lonRange)

v

In [ ]:
print(v.values)

As can be seen, the v-component wind data variable is in units of meters/second (m/s). It is better to convert this into knots (kts), because once again that is a more commonly measured unit for wind in weather maps. Luckily, we can use a simple unit conversion method to do this again. 

In [ ]:
v_kts = v.metpy.convert_units('kts')

In [ ]:
print(v_kts.values)

We once again need to select a pressure level. Let's use the 850 hPa level this time, where wind flow in weather maps is commonly measured. 

In [ ]:
target_pressure = 850 * units.hPa 
v_selected_level = v.sel(isobaricInhPa=target_pressure, method='nearest')

print("Shape after selecting 850 hPa level:", v_selected_level.shape)

Then we can finally make the plot, using the same setup we used for the vertical velocity, except making sure to modify the lat/lon extent to the new variables we defined, and the modified variable we used to convert the 'v' data variable into a 2D data shape. 

In [ ]:
datacrs = ccrs.PlateCarree()

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(1, 1, 1, projection=datacrs) 

ax.set_extent([lonmin, lonmax, latmin, latmax], crs=datacrs)
ax.add_feature(cfeature.STATES.with_scale('50m'), edgecolor='black', alpha=0.6)

gl = ax.gridlines(draw_labels=True, linewidth=1, color='gray', alpha=0.5, linestyle='--')
gl.top_labels = False
gl.right_labels = False

cf = ax.contourf(v_selected_level.longitude, v_selected_level.latitude, v_selected_level,
                 levels=15, cmap='coolwarm', transform=datacrs)

cbar = fig.colorbar(cf, ax=ax, orientation='vertical', 
                    label=f'V Wind Component (kts)') 

level_val = v_selected_level.isobaricInhPa.item() 
ax.set_title(f'V Wind Component at {level_val:.0f} hPa for {timeStr}')

plt.show()

On the graph, the red values represent positive values of the v-component (southerly wind), and the blues represent negative values (northerly wind). The system's warm sector over eastern NY and New England down into the Mid-Atlantic can clearly be made out, as well as the winds turning more northerly behind the cold frontal passage across the Great Lakes into the Ohio Valley. The front was moving through western NY and northwest PA at this time, which explains the lighter red and blue shadings as you go further west. 

## Summary

To conclude this notebook, it can be seen that you can use another dataset like ERA5 to verify the accuracy of the GridRad radar reflecitivy data, as well as other variables. In this notebook, we made graphs of the vertical pressure velocity at 700 hPA and the v-wind component at 850 hPa from the ERA5 Pressure Level Hourly Data, and this seemed to match up with the GR-S radar refl. and AZ shear data in regards to location, as well as real-life observations. 

## Resources and References

The data from the ERA5 hourly data on pressure levels to make the graphs is hosted by the Destination Earth Data Hub: https://earthdatahub.destine.eu/collections/era5/datasets/reanalysis-era5-pressure-levels 